# 05 — 多尺度对比分析

## 目标

对比 4 种网格尺度（1km / 500m / 200m / 100m）下供需分析的差异，回答：

1. **网格粗细对分析结果影响多大？** — 同一场景在不同尺度下短缺/富余的分布有何不同
2. **哪种尺度最适合调度建模？** — 结合模型结果（04 notebook），给出推荐
3. **OD 流在不同尺度下如何变化？** — 粗网格将细粒度 OD 聚合后的效果
4. **数据量随尺度增长多快？** — 网格数和 OD 对数的指数增长

## 数据来源
4 个尺度的 `scenario_grid_demand_*.parquet` 和 `od_flow_grid_*.parquet`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

SCALES = ['1km', '500m', '200m', '100m']
SCENARIO = '20210512_pm_peak'  # 基准场景

print('库加载完成')

In [ ]:
# 加载 4 个尺度的需求和 OD 流
demand_dict = {}
od_dict = {}
meta_dict = {}

for s in SCALES:
    demand_dict[s] = pd.read_parquet(f'../data/processed/scenario_grid_demand_{s}.parquet')
    od_dict[s] = pd.read_parquet(f'../data/processed/od_flow_grid_{s}.parquet')
    meta_dict[s] = pd.read_csv(f'../data/processed/grid_metadata_{s}.csv')

# 基本统计
stats = []
for s in SCALES:
    d = demand_dict[s]
    o = od_dict[s]
    sc_d = d[d['scenario_id'] == SCENARIO]
    sc_o = o[o['scenario_id'] == SCENARIO]
    stats.append({
        'Scale': f'{s} ({int(s.replace("km","").replace("m",""))}m)' if 'm' in s else f'{s} (1000m)',
        'Grids': sc_d['grid_id'].nunique(),
        'Shortage Grids': (sc_d['shortage'] > 0).sum(),
        'Surplus Grids': (sc_d['surplus'] > 0).sum(),
        'Total Shortage': sc_d['shortage'].sum(),
        'Total Surplus': sc_d['surplus'].sum(),
        'OD Pairs': len(sc_o),
        'OD Total Orders': sc_o['order_count'].sum(),
    })

stats_df = pd.DataFrame(stats)
stats_df

In [ ]:
# 可视化：网格数和 OD 对数随尺度的增长
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

scales_label = ['1km', '500m', '200m', '100m']
x = range(len(scales_label))

# 网格总数
axes[0].bar(x, stats_df['Grids'], color='#4575B4', edgecolor='white')
axes[0].set_xticks(x); axes[0].set_xticklabels(scales_label)
axes[0].set_title('Number of Grids\n(finer = more grids)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Grids')
for i, v in enumerate(stats_df['Grids']):
    axes[0].text(i, v + 100, f'{v:,}', ha='center', fontsize=9)

# 短缺/富余网格数
w = 0.35
axes[1].bar([i - w/2 for i in x], stats_df['Shortage Grids'], w, color='#D73027', edgecolor='white', label='Shortage')
axes[1].bar([i + w/2 for i in x], stats_df['Surplus Grids'], w, color='#1A9850', edgecolor='white', label='Surplus')
axes[1].set_xticks(x); axes[1].set_xticklabels(scales_label)
axes[1].set_title('Shortage vs Surplus Grids', fontsize=12, fontweight='bold')
axes[1].legend()

# OD 对数
axes[2].bar(x, stats_df['OD Pairs'], color='#F0A030', edgecolor='white')
axes[2].set_xticks(x); axes[2].set_xticklabels(scales_label)
axes[2].set_title('Number of OD Pairs\n(finer = more sparse)', fontsize=12, fontweight='bold')
for i, v in enumerate(stats_df['OD Pairs']):
    axes[2].text(i, v + 50, f'{v:,}', ha='center', fontsize=9)

fig.suptitle(f'Multi-Scale Data Comparison — {SCENARIO}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/05_scale_stats.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 1. 同一场景在不同尺度下的短缺/富余分布

### 这是什么

把 4 个尺度的短缺（红）和富余（绿）散点图放在一起，看空间模式的差异。

- 尺度越粗 → 每个网格覆盖更大区域 → 内部需求被平均化 → 失衡看起来更"平滑"
- 尺度越细 → 更能捕捉局部热点 → 但噪声也更大

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

for idx, s in enumerate(SCALES):
    ax = axes[idx]
    d = demand_dict[s]
    m = meta_dict[s]
    sc_d = d[d['scenario_id'] == SCENARIO].merge(m[['grid_id','center_lng','center_lat']], on='grid_id')

    short = sc_d[sc_d['shortage'] > 0]
    surp = sc_d[sc_d['surplus'] > 0]

    if len(short) > 0:
        ax.scatter(short['center_lng'], short['center_lat'],
                   s=np.sqrt(short['shortage'])*10, c='#D73027', alpha=0.5, edgecolors='#A50F15', linewidth=0.3)
    if len(surp) > 0:
        ax.scatter(surp['center_lng'], surp['center_lat'],
                   s=np.sqrt(surp['surplus'])*10, c='#1A9850', alpha=0.5, edgecolors='#006837', linewidth=0.3)

    ax.set_title(f'{s}\n{len(sc_d):,} grids, S={len(short)}, P={len(surp)}', fontsize=11, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([]); ax.set_aspect('equal')

fig.suptitle(f'Shortage vs Surplus Across 4 Scales — {SCENARIO}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'../outputs/figures/05_surplus_shortage_4scales.png', dpi=200, bbox_inches='tight')
plt.show()

---

## 2. 净流出分布对比 — 统一色阶

用统一色阶对比 4 个尺度的 net_outflow 热力图，看空间模式的精细度差异。

In [ ]:
from matplotlib.colors import TwoSlopeNorm

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

# 用 200m 尺度的色阶范围作为基准（最细粒度覆盖最广）
ref_all = demand_dict['200m']
global_vmax = max(abs(ref_all['net_outflow'].min()), abs(ref_all['net_outflow'].max()))

for idx, s in enumerate(SCALES):
    ax = axes[idx]
    d = demand_dict[s]
    m = meta_dict[s]
    sc_d = d[d['scenario_id'] == SCENARIO].merge(m[['grid_id','center_lng','center_lat']], on='grid_id')

    norm = TwoSlopeNorm(vmin=-global_vmax, vcenter=0, vmax=global_vmax)
    ax.scatter(sc_d['center_lng'], sc_d['center_lat'], s=1, c='#E8E8E8', alpha=0.3, edgecolors='none')
    ax.scatter(sc_d['center_lng'], sc_d['center_lat'],
               c=sc_d['net_outflow'], cmap='RdBu_r', norm=norm,
               s=abs(sc_d['net_outflow'])*1.5+1, alpha=0.7, edgecolors='none')

    ax.set_title(f'{s} ({len(sc_d):,} grids)', fontsize=11, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([]); ax.set_aspect('equal')

fig.suptitle(f'Net Outflow Comparison Across 4 Scales (Unified Color Scale) — {SCENARIO}',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'../outputs/figures/05_net_outflow_4scales.png', dpi=200, bbox_inches='tight')
plt.show()

---

## 3. 尺度聚合效应 — 细→粗的流失

### 这是什么

把 200m（最细）的需求按网格 ID 前缀聚合到 1km 尺度，对比与原生 1km 数据是否一致。验证数据管线中不同尺度的网格划分是否自洽。

In [ ]:
# 从 200m 网格 ID 提取 1km 父网格（去掉行号的精细部分）
# 200m: 200m_r032_c034 -> 1km 父: 1000m_r003_c003
d200 = demand_dict['200m']
d200_sc = d200[d200['scenario_id'] == SCENARIO].copy()

# 从 grid_id 提取 row/col 编号并映射到 1km
d200_sc['row'] = d200_sc['grid_id'].str.extract(r'_r(\d+)').astype(int)
d200_sc['col'] = d200_sc['grid_id'].str.extract(r'_c(\d+)').astype(int)

# 200m -> 1km: 每 5 个 200m 网格 = 1 个 1km 网格
d200_sc['km_row'] = (d200_sc['row'] // 5).astype(int)
d200_sc['km_col'] = (d200_sc['col'] // 5).astype(int)
d200_sc['km_grid'] = '1000m_r' + d200_sc['km_row'].astype(str).str.zfill(3) + '_c' + d200_sc['km_col'].astype(str).str.zfill(3)

# 聚合到 1km
agg_1km = d200_sc.groupby('km_grid').agg(
    departures=('departures','sum'),
    arrivals=('arrivals','sum')
).reset_index()
agg_1km['net_outflow'] = agg_1km['departures'] - agg_1km['arrivals']

# 原生 1km
d1 = demand_dict['1km']
d1_sc = d1[d1['scenario_id'] == SCENARIO].copy()

# 合并对比
compare_agg = agg_1km.merge(d1_sc[['grid_id','net_outflow']], left_on='km_grid', right_on='grid_id',
                             how='inner', suffixes=('_agg','_native'))

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(compare_agg['net_outflow_native'], compare_agg['net_outflow_agg'],
           c='#4575B4', alpha=0.5, s=20)
max_val = max(abs(compare_agg['net_outflow_native'].max()), abs(compare_agg['net_outflow_agg'].max()))
ax.plot([-max_val, max_val], [-max_val, max_val], 'r--', linewidth=1, label='y=x (perfect match)')
ax.set_xlabel('Native 1km net_outflow')
ax.set_ylabel('200m aggregated to 1km net_outflow')
ax.set_title('Aggregation Consistency Check\n200m -> 1km vs Native 1km', fontsize=12, fontweight='bold')
ax.set_aspect('equal'); ax.legend()
plt.tight_layout()
plt.savefig('../outputs/figures/05_aggregation_check.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 4. 模型结果对比

3 个尺度（1km / 500m / 200m）的模型求解结果汇总。

In [ ]:
model_scales = ['1000m', '500m', '200m']
model_data = []
for g in model_scales:
    s = pd.read_csv(f'../outputs/model_results/summary_{g}_{SCENARIO}.csv')
    o = pd.read_csv(f'../outputs/model_results/objective_parts_{g}_{SCENARIO}.csv')
    ss = pd.read_csv(f'../outputs/model_results/shortage_service_{g}_{SCENARIO}.csv')
    nc = pd.read_csv(f'../outputs/model_results/node_copies_{g}_{SCENARIO}.csv')

    # 计算整体满足率
    ss['ogrid'] = ss['grid_id'].map(dict(zip(nc['grid_id'], nc['original_grid_id']))).fillna(ss['grid_id'])
    rs = ss.groupby('ogrid').agg(s=('shortage_bikes','sum'), sv=('served_bikes','sum')).reset_index()
    sat_rate = rs['sv'].sum() / rs['s'].sum() if rs['s'].sum() > 0 else 0

    label = g.replace('1000m', '1km')
    model_data.append({
        'Scale': label,
        'Status': s['status'][0],
        'Objective': f"{s['objective'][0]:,.0f}",
        'Runtime': f"{s['runtime_seconds'][0]:.0f}s",
        'Makespan': f"{o['makespan_min'][0]:.0f}min",
        'Unmet': f"{o['total_unmet_bikes'][0]:.0f}",
        'Served': f"{o['total_served_bikes'][0]:.0f}",
        'Satisfaction': f"{sat_rate*100:.1f}%",
    })

pd.DataFrame(model_data)

---

## 关键发现与建议

1. **数据量爆炸增长**：从 1km 到 100m，网格数从 1,150 增长到 61,060（53 倍），OD 对从 8,660 增长到 46,370（5.4 倍）。100m 尺度数据量巨大但可视化可读性差。

2. **空间模式基本一致**：4 个尺度的短缺/富余热点位置相似，但细粒度能捕捉到更局部的失衡（如单个地铁站周边的微热点）。

3. **聚合自洽性**：200m→1km 与原生 1km 高度一致，说明数据管线的网格划分是自洽的。

4. **推荐 500m 为建模尺度**：
   - 1km 太粗：丢失局部细节，且 OD 聚合过度
   - 500m 是**平衡点**：足够细捕捉空间模式，求解可找到可行解（84.1% 满足率）
   - 200m 太细：691s 求解时间仍无法满足服务水平
   - 100m 仅用于可视化参考，不适合当前规模的优化求解

5. **可视化推荐 200m**：展示分析用 200m 尺度（约 200m×200m 网格 = 一个街区），可读性和信息量平衡最好。

## 下一步

→ 撰写最终报告，汇总 01-05 的所有关键发现，给出调度策略建议